In [2]:
import glob
data_path = "data/"
row_filenames = glob.glob(data_path + "/row/*")

In [ ]:
import numpy as np
import pandas as pd
from typing import List


def normalize_column(df: pd.DataFrame, x: str, y: List[str], new_x) -> pd.DataFrame:
    """
    Возвращает новый DataFrame с равномерно расположенными значениями колонки `x`
    от `x0` до `xmax` (включительно) с шагом `step`. Для каждой колонки в `y`
    выполняется линейная интерполяция по соседним не нулевым и не-null значениям.

    Особенности:
    - Значения `0` в колонках y считаются отсутствующими и игнорируются при интерполяции.
    - Если для колонки y нет ни одного ненулевого значения — в результирующем столбце будут NaN.
    - Если только одно ненулевое значение — оно будет подставлено только в точку, где оно встречается.

    Параметры:
    - df: входной DataFrame
    - x: имя колонки, которая рассматривается как координата/параметр (должна быть числовой)
    - y: список имён колонок, которые нужно интерполировать
    - x0: начальное значение для сетки
    - step: шаг сетки (целое положительное)
    - xmax: максимальное значение сетки
    """

    df2 = df.copy()
    # Преобразуем x в числовой тип и удалим строки с некорректным x
    df2[x] = pd.to_numeric(df2[x], errors='coerce')
    df2 = df2[df2[x].notna()]

    # Агрегируем по x на случай дублирующихся x (берём среднее по y)
    grouped = df2.groupby(x, as_index=True)[y].mean()

    # Результирующий DataFrame
    result = pd.DataFrame({x: new_x})

    # Для каждой y-колонки выполняем интерполяцию по числовому индексу
    for col in y:
        if col in grouped.columns:
            series = grouped[col].astype(float)
        else:
            # Если такой колонки нет во входном наборе — заполним NaN
            series = pd.Series(dtype='float64')

        # Считаем, что значения 0 и NaN — отсутствуют и не участвуют в интерполяции
        mask = series.notna() & (series != 0)

        if mask.sum() == 0:
            # Нет известных точек
            interp_vals = np.array([np.nan] * len(new_x), dtype=float)
        elif mask.sum() == 1:
            # Только одна точка — только в её x будет значение, в остальных NaN
            known_x = series[mask].index.to_numpy(dtype=float)
            known_y = series[mask].to_numpy(dtype=float)
            interp_vals = np.array([known_y[0] if xx == known_x[0] else np.nan for xx in new_x], dtype=float)
        else:
            known_x = series[mask].index.to_numpy(dtype=float)
            known_y = series[mask].to_numpy(dtype=float)
            interp_vals = np.interp(new_x.astype(float), known_x, known_y)
            left_mask = new_x < known_x.min()
            right_mask = new_x > known_x.max()
            interp_vals[left_mask | right_mask] = np.nan

        result[col] = interp_vals

    return result

def visualize_nulls(df:pd.DataFrame):
    import pandas as pd
    import seaborn as sns
    import matplotlib.pyplot as plt

    # Создание матрицы пропусков
    plt.figure(figsize=(12, 8))
    sns.heatmap(df.isnull(),
                cbar=False,           # убрать цветовую шкалу
                cmap='viridis',       # цветовая схема
                yticklabels=False)    # убрать метки строк
    plt.title('Матрица пропусков в данных')
    plt.show()


In [ ]:
import json
import pandas as pd
# row_filenames = [data_path + "row/pAnGwRiQ4-4_hourly.json"]

def convert_to_dataframe(data:list):
    df = pd.DataFrame(data)
    s = df['timestamp'].astype(str).str.strip()
    ts = pd.to_datetime(s, utc=True, errors='coerce', format='ISO8601')
    df['timestamp'] = ts.apply(lambda x: int(x.timestamp()) if pd.notnull(x) else pd.NA).astype("Int64")
    df.drop(["vph"], axis=1, inplace=True)
    return df

days = 10
hour = 60*60
hours_in_10_days = days*24


with open(data_path + "out.csv", "w") as out_csv:
    header = [
        "videoId",
        "timestamp",
    ]
    header.extend((a+f"_{i}" for i in range(1,hours_in_10_days+1) for a in ["comments","likes","views"]))

    out_csv.write(";".join(header) + "\n")

    for row_filename in row_filenames:
        print(row_filename)
        with open(row_filename, "r") as f:
            data = json.load(f)
        video_id = data["videoId"]


        df = convert_to_dataframe(data["data"])
        df.sort_values(by=["timestamp"], inplace=True)

        established_at = df["timestamp"].min()



        new_x = np.arange(established_at, established_at+hour*hours_in_10_days, hour)
        df:pd.DataFrame = normalize_column(df, x="timestamp", y=["views","likes","comments"], new_x=new_x)

        line = [
        video_id,
        established_at,
        ]

        for a in ["views","likes","comments"]:
            line.extend(df[a])

        out_csv.write((";".join(map(str,line)) + "\n").replace(".",","))


print("complited")



In [ ]:
import os
from itertools import chain
df = pd.read_csv(os.path.join(data_path, "out.csv"),delimiter=";")

visualize_nulls(df)

hours_for_drop = chain(range(25,192), range(193,241))
drop_names = ["likes_192","comments_192"]
for hour in hours_for_drop:
    for column_prefix in ("comments","views","likes"):
        drop_names.append(f"{column_prefix}_{hour}")
df.drop(drop_names, axis=1, inplace=True)



df.sort_values(["timestamp"], inplace=True)

visualize_nulls(df)


df.to_csv(os.path.join(data_path, "cuted.csv"), index=False)

In [18]:
# len(df)*0.6
import numpy as np
import os
import pandas as pd
df = pd.read_csv(os.path.join(data_path, "cuted.csv"))
print(list(df['views_192'].isna()).count(True))

df.dropna(subset=["views_192"], inplace=True)
print( list(df['views_192'].isna()).count(True))
# list(df[not df['views_192'].empty].isna()).count(False)

101
0


In [19]:
def create_rolling_dataset_advanced(df, target_col='views_192h', timestamp_col='timestamp'):
    """
    Создает rolling dataset с временными признаками и инженерией
    """
    X_list = []
    y_list = []
    hour_list = []
    post_id_list = []

    for post_id in df.index:
        target = df.loc[post_id, target_col]
        timestamp = df.loc[post_id, timestamp_col]

        # Обработка timestamp (автоопределение формата)
        try:
            # Пробуем как секунды
            dt = pd.to_datetime(timestamp, unit='s')
        except:
            try:
                # Пробуем как миллисекунды
                dt = pd.to_datetime(timestamp, unit='ms')
            except:
                # Пробуем как строку
                dt = pd.to_datetime(timestamp)


        for i in range(3, 25):  # i = текущий час (3..24)
            try:

                # Базовые значения
                views_lag2 = df.loc[post_id, f'views_{i-2}']
                views_lag1 = df.loc[post_id, f'views_{i-1}']
                views_curr = df.loc[post_id, f'views_{i}']

                likes_lag2 = df.loc[post_id, f'likes_{i-2}']
                likes_lag1 = df.loc[post_id, f'likes_{i-1}']
                likes_curr = df.loc[post_id, f'likes_{i}']

                comments_lag2 = df.loc[post_id, f'comments_{i-2}']
                comments_lag1 = df.loc[post_id, f'comments_{i-1}']
                comments_curr = df.loc[post_id, f'comments_{i}']


                features = {

                    'timestamp': timestamp,
                    'views_current': views_curr,
                    'views_lag1': views_lag1,
                    'views_lag2': views_lag2,
                    'likes_current': likes_curr,
                    'comments_current': comments_curr,


                }

                X_list.append(features)
                y_list.append(target)
                hour_list.append(i)
                post_id_list.append(post_id)

            except KeyError as e:
                print(f"Пропущен столбец для post_id={post_id}, час={i}: {e}")
                continue

    X = pd.DataFrame(X_list)
    y = np.array(y_list)

    return X, y, hour_list, post_id_list

In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score




def train_and_evaluate_model(df, target_col='views_192h', timestamp_col='timestamp',
                            test_size=0.2, model_type='ridge'):
    """
    Полный пайплайн: создание признаков, обучение, оценка

    Parameters:
    -----------
    df : DataFrame
        Исходные данные
    target_col : str
        Название целевой колонки
    timestamp_col : str
        Название колонки с timestamp
    test_size : float
        Доля тестовых данных (по времени)
    model_type : str
        'ridge' или 'random_forest'

    Returns:
    --------
    results : dict
        Словарь с результатами
    """

    print("=" * 60)
    print("ШАГ 1: Создание rolling dataset")
    print("=" * 60)

    # Создаем rolling dataset
    X, y, hours, post_ids = create_rolling_dataset_advanced(
        df, target_col, timestamp_col
    )

    print(f"Размер X: {X.shape}")
    print(f"Количество признаков: {X.shape[1]}")
    print(f"Всего примеров: {len(y)}")
    print(f"Диапазон часов: {min(hours)} - {max(hours)}")

    # Добавляем информацию о посте и часе в DataFrame для группировки
    X['post_id'] = post_ids
    X['hour'] = hours
    y_df = pd.DataFrame({'target': y})

    print("\n" + "=" * 60)
    print("ШАГ 2: Разделение на train/test по времени")
    print("=" * 60)

    # Разделяем по постам (чтобы все примеры одного поста были в одном сете)
    unique_posts = df.index.unique()
    n_train = int(len(unique_posts) * (1 - test_size))

    # Сортируем посты по timestamp для временного разделения
    posts_with_time = df[[timestamp_col]].copy()
    posts_with_time = posts_with_time.sort_values(timestamp_col)
    train_posts = posts_with_time.index[:n_train]
    test_posts = posts_with_time.index[n_train:]

    print(f"Train посты: {len(train_posts)}")
    print(f"Test посты: {len(test_posts)}")

    # Фильтруем
    train_mask = X['post_id'].isin(train_posts)
    test_mask = X['post_id'].isin(test_posts)

    X_train = X[train_mask].drop(['post_id', 'hour'], axis=1)
    X_test = X[test_mask].drop(['post_id', 'hour'], axis=1)
    y_train = y_df[train_mask]['target'].values
    y_test = y_df[test_mask]['target'].values
    hours_test = X[test_mask]['hour'].values

    print(f"Train размер: {X_train.shape}")
    print(f"Test размер: {X_test.shape}")

    # Логарифмируем целевую переменную
    y_train_log = np.log1p(y_train)
    y_test_log = np.log1p(y_test)

    print("\n" + "=" * 60)
    print(f"ШАГ 3: Обучение модели ({model_type})")
    print("=" * 60)

    # Выбираем модель
    if model_type == 'ridge':
        model = make_pipeline(
            StandardScaler(),
            RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], cv=5)
        )
        print("Используется Ridge Regression с CV")
    elif model_type == 'random_forest':
        model = RandomForestRegressor(
            n_estimators=100,
            max_depth=5,
            min_samples_split=10,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1
        )
        print("Используется Random Forest")
    else:
        raise ValueError("model_type должен быть 'ridge' или 'random_forest'")

    # Обучаем
    model.fit(X_train, y_train_log)

    # Предсказания
    y_pred_log = model.predict(X_test)
    y_pred = np.expm1(y_pred_log)  # обратное преобразование

    print("\n" + "=" * 60)
    print("ШАГ 4: Оценка качества")
    print("=" * 60)

    # Общие метрики
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print(f"\nОбщие метрики на тесте:")
    print(f"MAE: {mae:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"R²: {r2:.4f}")

    # Анализ по часам
    print("\n" + "=" * 60)
    print("ШАГ 5: Анализ качества по часам прогноза")
    print("=" * 60)

    results_by_hour = []
    for hour in sorted(set(hours_test)):
        mask = hours_test == hour
        if mask.sum() > 0:
            hour_mae = mean_absolute_error(y_test[mask], y_pred[mask])
            hour_r2 = r2_score(y_test[mask], y_pred[mask])
            results_by_hour.append({
                'hour': hour,
                'n_samples': mask.sum(),
                'mae': hour_mae,
                'r2': hour_r2
            })

    results_df = pd.DataFrame(results_by_hour)
    print(results_df.to_string(index=False))

    # График улучшения качества (опционально)
    print("\n" + "=" * 60)
    print("ШАГ 6: Важность признаков (для Random Forest)")
    print("=" * 60)

    if model_type == 'random_forest':
        feature_importance = pd.DataFrame({
            'feature': X_train.columns,
            'importance': model.feature_importances_
        }).sort_values('importance', ascending=False)

        print("\nТоп-10 важнейших признаков:")
        print(feature_importance.head(10).to_string(index=False))

    # Возвращаем результаты
    results = {
        'model': model,
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test,
        'y_pred': y_pred,
        'metrics': {
            'mae': mae,
            'rmse': rmse,
            'r2': r2
        },
        'results_by_hour': results_df,
        'hours_test': hours_test
    }

    if model_type == 'random_forest':
        results['feature_importance'] = feature_importance

    return results



In [17]:
# 1. Подготовьте ваши данные в формате DataFrame
# Ваш df должен содержать:
# - timestamp (epoch)
# - views_1 .. views_24
# - likes_1 .. likes_24
# - comments_1 .. comments_24
# - views_192h (целевая переменная)

# 2. Запустите функцию
df.head()
results = train_and_evaluate_model(
    df=df,
    target_col='views_192h',
    timestamp_col='timestamp',
    test_size=0.2,          # 20% на тест
    model_type='ridge'      # или 'random_forest'
)

# 3. Посмотрите результаты
print(f"MAE: {results['metrics']['mae']:.2f}")
print(f"R²: {results['metrics']['r2']:.4f}")
print(results['results_by_hour'])  # качество по часам

# # 4. Используйте модель для прогноза новых постов
# prediction = quick_predict(
#     model=results['model'],
#     post_data=new_post_data,
#     current_hour=6,
#     timestamp=current_timestamp,
#     feature_columns=results['X_train'].columns.tolist()
# )

ШАГ 1: Создание rolling dataset


KeyError: 'views_192h'